Ratings Dataset:
   userId  movieId  rating  timestamp
0       1        1     4.0  964982703
1       1        3     4.0  964981247
2       1        6     4.0  964982224
3       1       47     5.0  964983815
4       1       50     5.0  964982931

Original Movies Dataset:
   movieId                               title  \
0        1                    Toy Story (1995)   
1        2                      Jumanji (1995)   
2        3             Grumpier Old Men (1995)   
3        4            Waiting to Exhale (1995)   
4        5  Father of the Bride Part II (1995)   

                                         genres  
0  Adventure|Animation|Children|Comedy|Fantasy   
1                    Adventure|Children|Fantasy  
2                                Comedy|Romance  
3                          Comedy|Drama|Romance  
4                                        Comedy  

Movies Dataset after cleaning titles:
   movieId                        title  \
0        1                    Toy Story   
1  

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from math import sqrt

#Importing Dataset
ratings_df = pd.read_csv('ratings.csv')
print("Ratings Dataset:")
print(ratings_df.head())

#Importing the movies dataset
movies_df = pd.read_csv('movies.csv')
print("\nOriginal Movies Dataset:")
print(movies_df.head())

#DATA PREPROCESSING

# Extract year from title and assign it to a new column
movies_df['year'] = movies_df.title.str.extract('(\(\d\d\d\d\))', expand=True)

# Remove parentheses from year column to get clean year
movies_df['year'] = movies_df.year.str.extract('(\d\d\d\d)', expand=True)

# FIXED: Remove year from title completely
movies_df['title'] = movies_df.title.str.replace('\(\d\d\d\d\)', '', regex=True)

# Remove all whitespaces from title
movies_df['title'] = movies_df['title'].apply(lambda x: x.strip())

print("\nMovies Dataset after cleaning titles:")
print(movies_df.head())

# Convert Genres into a list
movies_df['genres'] = movies_df.genres.str.split('|')
print("\nAfter converting genres to lists:")
print(movies_df.head())

# One Hot Encoding of Genres
movies_copy = movies_df.copy()

for index, row in movies_df.iterrows():
    for genre in row['genres']:
        movies_copy.at[index, genre] = 1

print("\nAfter One-Hot Encoding (first few columns):")
print(movies_copy.head())

# Fill NaN values with 0 
movies_copy = movies_copy.fillna(0)

# Clean up ratings dataset
ratings_df = ratings_df.drop(['timestamp'], axis=1)
print("\nCleaned Ratings Dataset:")
print(ratings_df.head())

#CONTENT BASED RECOMMENDATION SYSTEM

# User Input for ratings
user_input = [
    {'title': 'Toy Story', 'rating': 4.5},
    {'title': 'Jumanji', 'rating': 8.5},
    {'title': 'GoldenEye', 'rating': 7.0},
    {'title': 'Batman Forever', 'rating': 6.0}
]

movies_input = pd.DataFrame(user_input)
print("\nUser Input:")
print(movies_input)

# Add movieID to user input
input_id = movies_df[movies_df['title'].isin(movies_input['title'].tolist())]
print("\nMatched movies from dataset:")
print(input_id[['movieId', 'title', 'year']])

# Merge the datasets
movies_input = pd.merge(input_id, movies_input, on='title')
print("\nMerged user input with movie IDs:")
print(movies_input)

# Drop unnecessary columns
movies_input = movies_input.drop(['genres', 'year'], axis=1)
print("\nCleaned user input:")
print(movies_input)

# Get genre information for user-rated movies
movies_user = movies_copy[movies_copy['movieId'].isin(movies_input['movieId'].tolist())]
movies_user = movies_user.reset_index(drop=True)

# Create Genre Table for user movies
UserGenreTable = movies_user.drop(['movieId', 'title', 'genres', 'year'], axis=1)
print("\nUser Genre Table shape:", UserGenreTable.shape)

# Calculate user profile using dot product
UserProfile = UserGenreTable.transpose().dot(movies_input['rating'])
print("\nUser Profile (top genres):")
print(UserProfile.sort_values(ascending=False).head(10))

# Create genre table for all movies
GenreTable = movies_copy.set_index(movies_copy['movieId'])
GenreTable = GenreTable.drop(['movieId', 'title', 'genres', 'year'], axis=1)

# Calculate recommendation scores
Recommendation_df = ((GenreTable * UserProfile).sum(axis=1)) / UserProfile.sum()
Recommendation_df = Recommendation_df.sort_values(ascending=False)

print("\nTop 10 Recommendation Scores:")
print(Recommendation_df.head(10))

# Create final recommendation table
RecommendationTable = movies_df.loc[movies_df['movieId'].isin(Recommendation_df.head(20).keys())]
RecommendationTable = RecommendationTable.sort_values('movieId')

print("\n" + "="*60)
print("FINAL MOVIE RECOMMENDATIONS")
print("="*60)
print(f"{'Rank':<4} {'Movie Title':<40} {'Year':<6} {'Genres'}")
print("-" * 80)

# Display recommendations in a clean format
for i, (idx, row) in enumerate(RecommendationTable.head(10).iterrows(), 1):
    title = row['title']
    year = row['year'] if pd.notna(row['year']) else 'N/A'
    genres = ', '.join(row['genres'][:3]) + ('...' if len(row['genres']) > 3 else '')
    print(f"{i:<4} {title:<40} {year:<6} {genres}")

print("\n" + "="*60)
print(f"Recommendations based on your ratings:")
for _, row in movies_input.iterrows():
    print(f"- {row['title']}: {row['rating']}/10")
print("="*60)